# 02 — Score all texts with the four toxicity classifiers

Runs each pretrained model **once** on `audit_tune`, `audit_test`, and HateCheck, and saves one toxicity score (0–1) per text to `scores/`. All later experiments (`04_experiments.ipynb`) will read these files, so no model has to be run again.

**Notes to running on Google Colab:**

Runtime → Change runtime type → T4 GPU. On a T4 the full run should take roughly 10–20 minutes. On a CPU it would take hours.

The files from `01_load_data.ipynb` must be in `data/processed/` (`audit_tune.parquet`, `audit_test.parquet`, `hatecheck.parquet`)
--> put the whole `FinalProject` folder in Google Drive and set `PROJECT_DIR` below.

If Colab disconnects each (model, dataset) result is saved as soon as it finishes, and finished ones are skipped on re-run. Just run all cells again.

In [ ]:
%pip install -q transformers pandas pyarrow scikit-learn tqdm

In [ ]:
import os

# --- Colab + Google Drive setup (skipped automatically when running locally) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/FA26 classes/CS74 - Sarah Preum/FinalProject/"
    os.chdir(PROJECT_DIR)
except ImportError:
    pass  # not on Colab: run from the project folder
print("working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
working directory: /content/drive/MyDrive/FA26 classes/CS74 - Sarah Preum/FinalProject


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
torch.manual_seed(SEED)

PROC = Path("data/processed")
SCORES = Path("scores")
RAW_SCORES = SCORES / "per_model"
RAW_SCORES.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64 if DEVICE == "cuda" else 16
MAX_LEN = 512  # longest input the models accept; longer comments are truncated

# Set to True for a quick check (200 rows per dataset) before the full run.
SMOKE_TEST = False

print("device:", DEVICE, "| torch", torch.__version__, "| transformers", transformers.__version__)
if DEVICE == "cpu":
    print("WARNING: no GPU found. The full run will be very slow; use Colab with a T4 GPU.")

device: cuda | torch 2.14.1+cu130 | transformers 5.22.1


## Models

None of the models return probabilities directly. Each returns raw, unbounded numbers (logits) one per label (e.g. `-3.2`, `4.1`). I then convert them into **one probability (0–1) that the text is toxic**, in `score_texts` below. The conversion depends on how each model was trained:

| Short name | Hugging Face ID | Raw output (logits) | Trained as | What **I** apply | Label kept |
|---|---|---|---|---|---|
| Unbiased-RoBERTa | `unitary/unbiased-toxic-roberta` | 16 logits, one per label | multi-label | **sigmoid** on the `toxicity` logit only | `toxicity` |
| Toxic-BERT | `unitary/toxic-bert` | 6 logits, one per label | multi-label | **sigmoid** on the `toxic` logit only | `toxic` |
| S-NLP RoBERTa | `s-nlp/roberta_toxicity_classifier` | 2 logits (`neutral`, `toxic`) | single-label | **softmax** over both logits, keep `toxic` | `toxic` |
| Cardiff-Offensive | `cardiffnlp/twitter-roberta-base-offensive` | 2 logits (`non-offensive`, `offensive`) | single-label | **softmax** over both logits, keep `offensive` | `offensive` |

**Sigmoid/Softmax choice**
- **Multi-label** models were trained to answer each label as its own yes/no question (a comment can be toxic *and* an insult *and* mention a group). Each logit is turned into a probability on its own with a **sigmoid**, so the probabilities don't add up to 1. We only need the `toxicity`/`toxic` one.
- **Single-label** models were trained to pick exactly one class out of two. A **softmax** turns both logits into probabilities that add up to 1, and we keep the toxic/offensive one.

Using the same function the model was trained with makes our scores match what the model was designed to output.

The label names were checked against each model's `config.json` (`id2label`). The `load_model` function re-checks them and stops if anything has changed.

Texts are passed in **as-is** (no lowercasing or cleaning). The audit tests the models on real text, and each model has its own tokenizer.

In [ ]:
MODELS = {
    # short name: (Hugging Face ID, label to read, activation)
    "Unbiased-RoBERTa":  ("unitary/unbiased-toxic-roberta",            "toxicity",  "sigmoid"),
    "Toxic-BERT":        ("unitary/toxic-bert",                        "toxic",     "sigmoid"),
    "S-NLP RoBERTa":     ("s-nlp/roberta_toxicity_classifier",         "toxic",     "softmax"),
    "Cardiff-Offensive": ("cardiffnlp/twitter-roberta-base-offensive", "offensive", "softmax"),
}

DATASETS = {
    # name: (file, id column, text column)
    "audit_tune": (PROC / "audit_tune.parquet", "id", "comment_text"),
    "audit_test": (PROC / "audit_test.parquet", "id", "comment_text"),
    "hatecheck":  (PROC / "hatecheck.parquet", "case_id", "test_case"),
}

In [ ]:
data = {}
for name, (path, id_col, text_col) in DATASETS.items():
    df = pd.read_parquet(path)
    if SMOKE_TEST:
        df = df.sample(200, random_state=SEED)
    df = df[[id_col, text_col]].rename(columns={id_col: "id", text_col: "text"})
    df["text"] = df["text"].fillna("")   # 2 comments have missing text
    data[name] = df.reset_index(drop=True)
    print(f"{name}: {len(df):,} texts")

audit_tune: 21,148 texts
audit_test: 21,147 texts
hatecheck: 3,728 texts


## Scoring function

In [ ]:
def load_model(hf_id, label):
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(hf_id).to(DEVICE).eval()
    label2id = {v: int(k) for k, v in model.config.id2label.items()}
    assert label in label2id, f"{hf_id}: label '{label}' not in {model.config.id2label}"
    return tok, model, label2id[label]


@torch.no_grad()
def score_texts(texts, tok, model, label_idx, activation):
    # Sort by length so each batch has similar lengths (less padding = much faster),
    # then put the scores back in the original order.
    order = np.argsort([len(t) for t in texts])
    scores = np.empty(len(texts), dtype=np.float32)
    for start in tqdm(range(0, len(texts), BATCH_SIZE), leave=False):
        idx = order[start:start + BATCH_SIZE]
        batch = tok([texts[i] for i in idx], padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logits = model(**batch).logits.float()
        if activation == "sigmoid":
            probs = torch.sigmoid(logits[:, label_idx])
        else:
            probs = torch.softmax(logits, dim=-1)[:, label_idx]
        scores[idx] = probs.cpu().numpy()
    return scores

## Quick sanity check

Before the long run: each model should give a low score to the first sentence and a high score to the second. If not, the label mapping is wrong.

When S-NLP RoBERTa loads, `transformers` prints a "LOAD REPORT" saying `roberta.pooler.dense.*` is UNEXPECTED. That's harmless: the checkpoint contains an extra layer that the classification head doesn't use.

In [ ]:
CHECK = ["Thank you for sharing, this was a really helpful article.",
         "You are a pathetic idiot and everyone hates you."]
for short, (hf_id, label, act) in MODELS.items():
    tok, model, idx = load_model(hf_id, label)
    s = score_texts(CHECK, tok, model, idx, act)
    print(f"{short:18s} polite: {s[0]:.3f}   insult: {s[1]:.3f}")
    del model

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Unbiased-RoBERTa   polite: 0.000   insult: 0.997


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Toxic-BERT         polite: 0.001   insult: 0.992


... (3 outputs left out)

S-NLP RoBERTa      polite: 0.000   insult: 1.000


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Cardiff-Offensive  polite: 0.039   insult: 0.933


## Score everything

In [ ]:
model_info = {}
for short, (hf_id, label, act) in MODELS.items():
    tok, model, idx = load_model(hf_id, label)
    model_info[short] = {"hf_id": hf_id, "label": label, "activation": act,
                         "revision": getattr(model.config, "_commit_hash", None)}
    for ds_name, df in data.items():
        out = RAW_SCORES / f"{ds_name}__{short.replace(' ', '_')}{'__smoke' if SMOKE_TEST else ''}.parquet"
        if out.exists():
            print(f"skip (already done): {short} on {ds_name}")
            continue
        t0 = time.time()
        s = score_texts(df["text"].tolist(), tok, model, idx, act)
        pd.DataFrame({"id": df["id"], "score": s}).to_parquet(out)
        print(f"{short:18s} {ds_name:11s} {len(df):>6,} texts  {time.time() - t0:6.0f}s")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/331 [00:00<?, ?it/s]

Unbiased-RoBERTa   audit_tune  21,148 texts      54s


  0%|          | 0/331 [00:00<?, ?it/s]

Unbiased-RoBERTa   audit_test  21,147 texts      36s


  0%|          | 0/59 [00:00<?, ?it/s]

... (19 outputs left out)

  0%|          | 0/331 [00:00<?, ?it/s]

Cardiff-Offensive  audit_test  21,147 texts      36s


  0%|          | 0/59 [00:00<?, ?it/s]

Cardiff-Offensive  hatecheck    3,728 texts       1s


## Combine into one file per dataset

`scores/<dataset>.parquet` has one row per text: the id, the gold label, and one score column per model. These are the files `04_experiments.ipynb` will read.

In [ ]:
suffix = "__smoke" if SMOKE_TEST else ""
for ds_name, (path, id_col, _) in DATASETS.items():
    gold = pd.read_parquet(path).rename(columns={id_col: "id"})
    combined = gold[gold["id"].isin(data[ds_name]["id"])].copy()
    for short in MODELS:
        s = pd.read_parquet(RAW_SCORES / f"{ds_name}__{short.replace(' ', '_')}{suffix}.parquet")
        combined = combined.merge(s.rename(columns={"score": short}), on="id", how="left")
    assert combined[list(MODELS)].notna().all().all(), "some texts are missing a score"
    combined.to_parquet(SCORES / f"{ds_name}{suffix}.parquet")
    print(f"saved scores/{ds_name}{suffix}.parquet  {combined.shape}")

# Record exact model versions and settings (for the README / reproducibility)
with open(SCORES / f"model_info{suffix}.json", "w") as f:
    json.dump({"models": model_info, "max_len": MAX_LEN, "seed": SEED,
               "torch": torch.__version__, "transformers": transformers.__version__}, f, indent=2)

saved scores/audit_tune.parquet  (21148, 17)
saved scores/audit_test.parquet  (21147, 17)
saved scores/hatecheck.parquet  (3728, 17)


## Check the results

ROC-AUC on `audit_tune` (Civil Comments) and HateCheck
--> **simple check** that the scores make sense (AUC near or below 0.5 would mean something is wrong). The real evaluation will happen on `audit_test` in `04_experiments.ipynb`.

In [ ]:
tune = pd.read_parquet(SCORES / f"audit_tune{suffix}.parquet")
hc = pd.read_parquet(SCORES / f"hatecheck{suffix}.parquet")
pd.DataFrame({
    "AUC audit_tune": [roc_auc_score(tune["toxic"], tune[m]) for m in MODELS],
    "AUC HateCheck": [roc_auc_score(hc["hateful"], hc[m]) for m in MODELS],
    "mean score audit_tune": [tune[m].mean() for m in MODELS],
}, index=list(MODELS)).round(3)

,AUC audit_tune,AUC HateCheck,mean score audit_tune
Unbiased-RoBERTa,0.949,0.645,0.127
Toxic-BERT,0.903,0.637,0.082
S-NLP RoBERTa,0.958,0.722,0.123
Cardiff-Offensive,0.895,0.566,0.323


# Testing whether everything ran smoothly

In [ ]:
import pandas as pd, json
from pathlib import Path

EXPECTED = {"audit_tune": 21_148, "audit_test": 21_147, "hatecheck": 3_728}
MODEL_COLS = ["Unbiased-RoBERTa", "Toxic-BERT", "S-NLP RoBERTa", "Cardiff-Offensive"]
ok = lambda c: "PASS" if c else "FAIL"

print(ok(not SMOKE_TEST), "full run (SMOKE_TEST is False)")

# 1. 12 per-model files (4 models x 3 datasets), each with the right row count
for ds, n in EXPECTED.items():
    for m in MODEL_COLS:
        f = Path("scores/per_model") / f"{ds}__{m.replace(' ', '_')}.parquet"
        rows = len(pd.read_parquet(f)) if f.exists() else 0
        print(ok(rows == n), f"{f.name}: {rows:,} rows (expected {n:,})")

# 2. 3 combined files: right size, one column per model, no missing scores, all scores in [0, 1]
for ds, n in EXPECTED.items():
    df = pd.read_parquet(f"scores/{ds}.parquet")
    s = df[MODEL_COLS]
    print(ok(len(df) == n), f"scores/{ds}.parquet: {len(df):,} rows")
    print(ok(s.notna().all().all()), "  no missing scores")
    print(ok(((s >= 0) & (s <= 1)).all().all()), "  all scores between 0 and 1")
    print(ok(df["id"].is_unique), "  no duplicate ids")

# 3. model versions recorded
print(ok(Path("scores/model_info.json").exists()), "scores/model_info.json saved")

PASS full run (SMOKE_TEST is False)
PASS audit_tune__Unbiased-RoBERTa.parquet: 21,148 rows (expected 21,148)
PASS audit_tune__Toxic-BERT.parquet: 21,148 rows (expected 21,148)
PASS audit_tune__S-NLP_RoBERTa.parquet: 21,148 rows (expected 21,148)
PASS audit_tune__Cardiff-Offensive.parquet: 21,148 rows (expected 21,148)
PASS audit_test__Unbiased-RoBERTa.parquet: 21,147 rows (expected 21,147)
PASS audit_test__Toxic-BERT.parquet: 21,147 rows (expected 21,147)
PASS audit_test__S-NLP_RoBERTa.parquet: 21,147 rows (expected 21,147)
PASS audit_test__Cardiff-Offensive.parquet: 21,147 rows (expected 21,147)
PASS hatecheck__Unbiased-RoBERTa.parquet: 3,728 rows (expected 3,728)
PASS hatecheck__Toxic-BERT.parquet: 3,728 rows (expected 3,728)
PASS hatecheck__S-NLP_RoBERTa.parquet: 3,728 rows (expected 3,728)
PASS hatecheck__Cardiff-Offensive.parquet: 3,728 rows (expected 3,728)
PASS scores/audit_tune.parquet: 21,148 rows
PASS   no missing scores
PASS   all scores between 0 and 1
PASS   no duplicate i